# WhatsApp Behavioural Profiling — Google Colab Notebook

**Project:** Behavioural Profiling of WhatsApp Conversation History for Impersonation and Account Takeover Detection Using Machine Learning.

Run the cells from top to bottom. The final cell launches an interactive Gradio dashboard from Colab, so you do not need Hugging Face or Streamlit hosting.

**Research limitation:** The labelled data generated here is synthetic. Model scores demonstrate the workflow and do not establish real-world WhatsApp account-takeover detection accuracy. An alert is not proof of compromise. Do not upload private WhatsApp messages or identifiable personal data.

The original WAPI CSV has very few rows and no verified takeover labels or all five dashboard features, so it cannot be used directly to validate account-takeover detection.


**False-alert reduction:** this version requires agreement from both models and an RF suspicious-class score of at least 0.80 before displaying a review alert. Real WhatsApp exports are still experimental because the models were trained on synthetic data.

## 1. Install and import libraries

In [ ]:
%pip -q install gradio joblib scikit-learn pandas numpy matplotlib

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, average_precision_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
WORK_DIR = Path("/content/whatsapp_profiling")
WORK_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "avg_message_length",
    "messages_per_hour",
    "writing_style_similarity",
    "contact_change_ratio",
    "unusual_hour_ratio",
]
print("Libraries loaded. Working folder:", WORK_DIR)


## 2. Generate a labelled synthetic dataset

In [ ]:
# Simulated teaching data only; these are not real WhatsApp records.
rng = np.random.default_rng(RANDOM_STATE)
rows = []
for user_num in range(100):
    user_id = f"user_{user_num:03d}"
    baseline_length = rng.normal(42, 8)
    baseline_rate = max(3, rng.normal(8, 2))
    baseline_style = np.clip(rng.normal(0.82, 0.08), 0.45, 0.99)

    for session_num in range(20):
        label = int(rng.random() < 0.20)
        if label == 0:
            avg_length = np.clip(rng.normal(baseline_length, 6), 1, 150)
            rate = np.clip(rng.normal(baseline_rate, 2), 0, 60)
            style = np.clip(rng.normal(baseline_style, 0.08), 0, 1)
            contact_change = np.clip(rng.beta(1.2, 12), 0, 1)
            unusual_hour = np.clip(rng.beta(1.5, 8), 0, 1)
        else:
            avg_length = np.clip(rng.normal(90, 18), 1, 200)
            rate = np.clip(rng.normal(24, 7), 0, 100)
            style = np.clip(rng.normal(0.35, 0.15), 0, 1)
            contact_change = np.clip(rng.beta(5, 2), 0, 1)
            unusual_hour = np.clip(rng.beta(5, 2), 0, 1)

        rows.append({
            "user_id": user_id,
            "session_id": f"{user_id}_session_{session_num:02d}",
            "avg_message_length": round(float(avg_length), 3),
            "messages_per_hour": round(float(rate), 3),
            "writing_style_similarity": round(float(style), 3),
            "contact_change_ratio": round(float(contact_change), 3),
            "unusual_hour_ratio": round(float(unusual_hour), 3),
            "label": label,
        })

synthetic_df = pd.DataFrame(rows)
dataset_path = WORK_DIR / "synthetic_labelled_sessions.csv"
synthetic_df.to_csv(dataset_path, index=False)
print("Dataset shape:", synthetic_df.shape)
display(synthetic_df["label"].value_counts().rename_axis("label").to_frame("sessions"))
display(synthetic_df.head())
print("Saved:", dataset_path)


## 3. Split by user and train both models

In [ ]:
X = synthetic_df[FEATURES].copy()
y = synthetic_df["label"].astype(int)
groups = synthetic_df["user_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print("Training sessions:", len(X_train))
print("Test sessions:", len(X_test))
display(y_test.value_counts().rename_axis("test label").to_frame("sessions"))

rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )),
])
rf_model.fit(X_train, y_train)

iso_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", IsolationForest(
        n_estimators=200, contamination=0.20, random_state=RANDOM_STATE
    )),
])
iso_model.fit(X_train.loc[y_train == 0])
print("Random Forest and Isolation Forest trained.")


## 4. Evaluate model performance

In [ ]:
rf_pred = rf_model.predict(X_test).astype(int)
rf_prob = rf_model.predict_proba(X_test)
rf_classes = list(rf_model.classes_)
rf_score = rf_prob[:, rf_classes.index(1)]

iso_raw = iso_model.predict(X_test)
iso_pred = (iso_raw == -1).astype(int)
iso_score = -iso_model.decision_function(X_test)

def evaluate_model(name, truth, pred):
    print("\\n---", name, "---")
    print("Accuracy :", round(accuracy_score(truth, pred), 4))
    print("Precision:", round(precision_score(truth, pred, zero_division=0), 4))
    print("Recall   :", round(recall_score(truth, pred, zero_division=0), 4))
    print("F1 score :", round(f1_score(truth, pred, zero_division=0), 4))
    print(classification_report(truth, pred, zero_division=0))
    cm = confusion_matrix(truth, pred, labels=[0, 1])
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(cm)
    return cm

rf_cm = evaluate_model("Random Forest", y_test, rf_pred)
iso_cm = evaluate_model("Isolation Forest", y_test, iso_pred)

if y_test.nunique() == 2:
    print("RF ROC-AUC:", round(roc_auc_score(y_test, rf_score), 4))
    print("RF PR-AUC:", round(average_precision_score(y_test, rf_score), 4))
    print("Isolation Forest ROC-AUC:", round(roc_auc_score(y_test, iso_score), 4))
    print("Isolation Forest PR-AUC:", round(average_precision_score(y_test, iso_score), 4))

print("\\nThese are synthetic-data results, not real-world validation.")


## 5. Plot confusion matrices and compare models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm, title in [
    (axes[0], rf_cm, "Random Forest"),
    (axes[1], iso_cm, "Isolation Forest"),
]:
    ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1], ["Normal", "Suspicious"])
    ax.set_yticks([0, 1], ["Normal", "Suspicious"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
fig.tight_layout()
plt.show()

comparison = pd.DataFrame({
    "Model": ["Random Forest", "Isolation Forest"],
    "Accuracy": [accuracy_score(y_test, rf_pred), accuracy_score(y_test, iso_pred)],
    "Precision": [precision_score(y_test, rf_pred, zero_division=0), precision_score(y_test, iso_pred, zero_division=0)],
    "Recall": [recall_score(y_test, rf_pred, zero_division=0), recall_score(y_test, iso_pred, zero_division=0)],
    "F1": [f1_score(y_test, rf_pred, zero_division=0), f1_score(y_test, iso_pred, zero_division=0)],
})
display(comparison.round(4))
comparison.set_index("Model").plot(kind="bar", figsize=(9, 5), ylim=(0, 1.05))
plt.title("Model comparison on synthetic test sessions")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Save the trained models

In [ ]:
rf_path = WORK_DIR / "whatsapp_takeover_rf_model.joblib"
iso_path = WORK_DIR / "whatsapp_isolation_forest_model.joblib"
metrics_path = WORK_DIR / "model_comparison.csv"

joblib.dump(rf_model, rf_path)
joblib.dump(iso_model, iso_path)
comparison.to_csv(metrics_path, index=False)

print("Saved:")
print(rf_path)
print(iso_path)
print(metrics_path)


## 7. Build reusable CSV screening functions

In [ ]:
def validate_features(data):
    missing = [col for col in FEATURES if col not in data.columns]
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))
    X = data[FEATURES].copy()
    for col in FEATURES:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    if X.isna().any().any():
        raise ValueError("Required features contain missing or non-numeric values.")
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError("Features must contain finite numeric values.")
    if (X["avg_message_length"] < 0).any():
        raise ValueError("avg_message_length cannot be negative.")
    if (X["messages_per_hour"] < 0).any():
        raise ValueError("messages_per_hour cannot be negative.")
    for col in ["writing_style_similarity", "contact_change_ratio", "unusual_hour_ratio"]:
        if not X[col].between(0, 1).all():
            raise ValueError(f"{col} must be between 0 and 1.")
    return X

# Conservative screening settings:
# 1) Random Forest confidence must be at least 0.80.
# 2) Isolation Forest must also flag the same row.
# This reduces alerts caused by only one model, but it is NOT a calibrated
# probability and must be evaluated on representative labelled data.
RF_ALERT_THRESHOLD = 0.80

def detect_multiple_sessions(sessions_df):
    if sessions_df.empty:
        raise ValueError("The uploaded CSV contains no rows.")
    X = validate_features(sessions_df)

    rf_pred = rf_model.predict(X).astype(int)
    probabilities = rf_model.predict_proba(X)
    rf_score = probabilities[:, list(rf_model.classes_).index(1)]
    iso_flag = (iso_model.predict(X) == -1).astype(int)
    anomaly_score = -iso_model.decision_function(X)

    # Require agreement from BOTH models, plus high RF score, before showing
    # the stronger review alert. This is a conservative heuristic, not proof.
    flagged = (rf_score >= RF_ALERT_THRESHOLD) & (iso_flag == 1)

    results = sessions_df.copy()
    results["random_forest_prediction"] = rf_pred
    results["random_forest_score"] = np.round(rf_score, 4)
    results["isolation_forest_anomaly"] = iso_flag
    results["anomaly_score"] = np.round(anomaly_score, 4)
    results["screening_status"] = np.where(
        flagged,
        "Flagged for review (both models agree)",
        "No consensus alert (not proof of safety)"
    )
    return results

example = pd.DataFrame([{
    "avg_message_length": 95,
    "messages_per_hour": 24,
    "writing_style_similarity": 0.35,
    "contact_change_ratio": 0.70,
    "unusual_hour_ratio": 0.80,
}])
display(detect_multiple_sessions(example))

## 8. Launch the interactive dashboard

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import re
from datetime import datetime

# WhatsApp export formats vary by phone language and locale.
# These patterns cover common Android/iPhone exports in day/month/year and month/day/year forms.
MESSAGE_PATTERNS = [
    re.compile(r"^\[(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}),\s*(\d{1,2}:\d{2}(?::\d{2})?\s*(?:AM|PM|am|pm)?)\]\s*([^:]+?):\s?(.*)$"),
    re.compile(r"^(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}),\s*(\d{1,2}:\d{2}(?::\d{2})?\s*(?:AM|PM|am|pm)?)\s*-\s*([^:]+?):\s?(.*)$"),
]

def parse_whatsapp_export(text):
    """Parse common WhatsApp .txt exports. Returns message rows with timestamp/sender/text."""
    parsed = []
    current = None

    def parse_dt(date_str, time_str):
        stamp = f"{date_str} {time_str.strip()}"
        formats = [
            "%d/%m/%y %H:%M", "%d/%m/%Y %H:%M",
            "%m/%d/%y %H:%M", "%m/%d/%Y %H:%M",
            "%d/%m/%y %H:%M:%S", "%d/%m/%Y %H:%M:%S",
            "%m/%d/%y %H:%M:%S", "%m/%d/%Y %H:%M:%S",
            "%d/%m/%y %I:%M %p", "%d/%m/%Y %I:%M %p",
            "%m/%d/%y %I:%M %p", "%m/%d/%Y %I:%M %p",
            "%d/%m/%y %I:%M:%S %p", "%d/%m/%Y %I:%M:%S %p",
            "%m/%d/%y %I:%M:%S %p", "%m/%d/%Y %I:%M:%S %p",
        ]
        for fmt in formats:
            try:
                return datetime.strptime(stamp, fmt)
            except ValueError:
                continue
        return None

    for line in text.replace("\ufeff", "").splitlines():
        match = None
        for pattern in MESSAGE_PATTERNS:
            match = pattern.match(line)
            if match:
                break
        if match:
            if current:
                parsed.append(current)
            date_str, time_str, sender, body = match.groups()
            timestamp = parse_dt(date_str, time_str)
            current = {
                "timestamp": timestamp,
                "sender": sender.strip(),
                "message": body.strip(),
            }
        elif current:
            # Lines continuing a message are appended; non-message system notices
            # before the first message are ignored.
            current["message"] += "\n" + line
    if current:
        parsed.append(current)

    # Drop rows where the date format could not be parsed.
    parsed = [row for row in parsed if row["timestamp"] is not None]
    if not parsed:
        raise ValueError(
            "No messages were recognised. Upload the original WhatsApp .txt export "
            "with timestamps and sender names. Some language/locale formats may need "
            "a parser adjustment."
        )
    return pd.DataFrame(parsed)


def whatsapp_txt_to_features(chat_text):
    """Create one approximate feature row per sender from a WhatsApp text export."""
    messages = parse_whatsapp_export(chat_text)
    messages = messages.sort_values("timestamp").reset_index(drop=True)
    messages["message_length"] = messages["message"].fillna("").str.len()

    rows = []
    for sender, group in messages.groupby("sender", sort=False):
        group = group.sort_values("timestamp")
        times = group["timestamp"]
        duration_hours = max((times.max() - times.min()).total_seconds() / 3600, 1 / 60)
        msg_lengths = group["message_length"].astype(float)
        # This is only a rough consistency proxy, not a validated authorship measure.
        mean_len = float(msg_lengths.mean()) if len(msg_lengths) else 0.0
        cv = float(msg_lengths.std(ddof=0) / mean_len) if mean_len > 0 else 1.0
        style_proxy = float(np.clip(1.0 - cv, 0.0, 1.0))
        hours = times.dt.hour
        unusual_ratio = float(((hours < 6) | (hours >= 22)).mean())
        rows.append({
            "sender": sender,
            "message_count": int(len(group)),
            "avg_message_length": round(mean_len, 3),
            "messages_per_hour": round(float(len(group) / duration_hours), 3),
            "writing_style_similarity": round(style_proxy, 3),
            # A single exported chat does not reveal contact-change history.
            # Zero is a neutral placeholder, not a measured contact-change rate.
            "contact_change_ratio": 0.0,
            "unusual_hour_ratio": round(unusual_ratio, 3),
        })
    return pd.DataFrame(rows)


def gradio_analyse(file_path):
    if not file_path:
        raise gr.Error("Please upload a CSV file or a WhatsApp exported .txt file.")

    try:
        if file_path.lower().endswith(".txt"):
            with open(file_path, "r", encoding="utf-8-sig", errors="replace") as f:
                chat_text = f.read()
            features = whatsapp_txt_to_features(chat_text)
            source_note = (
                "TXT export parsed. Features are approximate proxies. "
                "Contact-change history is unavailable in an exported chat."
            )
            results = detect_multiple_sessions(features)
            results.insert(0, "input_type", "WhatsApp TXT export")
        else:
            data = pd.read_csv(file_path)
            results = detect_multiple_sessions(data)
            source_note = "CSV behavioural features screened."
    except Exception as exc:
        raise gr.Error(str(exc))

    flagged = int(results["screening_status"].str.startswith("Flagged for review").sum())
    total = len(results)
    summary = pd.DataFrame({
        "Metric": [
            "Rows / sender profiles screened",
            "Flagged for review",
            "No consensus alert",
            "Input note",
        ],
        "Value": [total, flagged, total - flagged, source_note],
    })

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(["Flagged for review", "No model alert"], [flagged, total - flagged])
    ax.set_title("Behavioural Screening Summary")
    ax.set_ylabel("Number of rows / sender profiles")
    ax.tick_params(axis="x", rotation=10)
    fig.tight_layout()

    output_path = WORK_DIR / "screened_behavioural_sessions.csv"
    results.to_csv(output_path, index=False)
    return summary, results.head(100), fig, str(output_path)


with gr.Blocks(title="WhatsApp Behavioural Profiling") as demo:
    gr.Markdown(
        "# WhatsApp Behavioural Profiling System\n"
        "### Impersonation and Account Takeover Screening\n\n"
        "Upload either a behavioural-features CSV or a WhatsApp exported chat `.txt` file. "
        "TXT exports are parsed into approximate sender-level features. The conservative rule flags only when BOTH models agree and Random Forest score is at least 0.80.\n\n"
        "**Research prototype:** the models were trained on synthetic data. "
        "TXT-derived feature proxies are not calibrated or validated for real-world "
        "takeover detection. Contact-change history cannot be determined from one "
        "exported chat. A no-alert result is not proof of safety. A review alert is not proof of compromise. Real TXT-derived features are not validated against these synthetic training data, so results are experimental. Avoid uploading private "
        "or identifiable conversations unless you have appropriate consent."
    )
    upload = gr.File(
        label="Upload CSV or WhatsApp exported chat TXT",
        file_types=[".csv", ".txt"],
        type="filepath",
    )
    analyse = gr.Button("Analyse sessions / chat", variant="primary")
    summary_out = gr.Dataframe(label="Screening Summary", interactive=False)
    results_out = gr.Dataframe(label="Screening Results", interactive=False)
    chart_out = gr.Plot(label="Screening Overview")
    download_out = gr.File(label="Download results CSV")
    analyse.click(
        fn=gradio_analyse,
        inputs=upload,
        outputs=[summary_out, results_out, chart_out, download_out],
    )

demo.launch(share=True, debug=True)


## 9. Test the dashboard with CSV or exported WhatsApp chat

When the last cell starts, open the temporary Gradio URL it prints.

### Option A — Upload a WhatsApp exported chat
1. Open the chat in WhatsApp.
2. Choose **More / Export chat** (menu wording varies by device).
3. Choose **Without media** to create a `.txt` file.
4. Transfer that file to your computer if needed.
5. In the dashboard, upload the `.txt` file and click **Analyse sessions / chat**.

The parser supports common WhatsApp timestamp formats such as `12/06/2026, 08:30 - Sender: message` and `[12/06/2026, 8:30 PM] Sender: message`. Export formats vary by device, language, and locale, so some files may need parser adjustments.

For TXT exports, the notebook creates approximate sender-level features from message timestamps and lengths. A single chat export does **not** provide contact-change history, so `contact_change_ratio` is set to a neutral placeholder of `0.0`. Writing-style similarity is only a rough message-length consistency proxy. Because the models were trained on synthetic data, their alerts/scores on exported chats are experimental and not validated evidence of impersonation or account takeover.

### Option B — Upload a behavioural-feature CSV
The CSV must contain:
- `avg_message_length` (non-negative)
- `messages_per_hour` (non-negative)
- `writing_style_similarity` (0–1)
- `contact_change_ratio` (0–1)
- `unusual_hour_ratio` (0–1)

The notebook-generated `synthetic_labelled_sessions.csv` is ready to test. The original WAPI CSV does not directly contain all five features, so it cannot be uploaded as-is.

To download the generated sample dataset from Colab:
```python
from google.colab import files
files.download(str(dataset_path))
```

To download trained models:
```python
from google.colab import files
files.download(str(rf_path))
files.download(str(iso_path))
```

The Gradio link is temporary and only works while the Colab runtime remains active. Do not upload private conversations or identifiable personal data without appropriate consent.


### False-alert reduction update
This notebook uses a **conservative consensus rule**: a row is flagged only when the Random Forest suspicious-class score is at least **0.80** AND Isolation Forest also flags it. This is intended to reduce one-model false alarms. It is a heuristic, not a validated real-world threshold. A normal chat may still be flagged because this project was trained on synthetic data and does not yet compare a person with a trusted historical baseline.

For a defensible real-world evaluation, collect consented, representative labelled examples, define a sender's normal baseline over multiple chats/sessions, and tune the threshold on a separate validation set. A single exported chat cannot establish whether an account was taken over.
